In [9]:
!rocm-smi



======================================== ROCm System Management Interface ========================================
================================================== Concise Info ==================================================
Device  Node  IDs              Temp    Power  Partitions          SCLK  MCLK   Fan    Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Edge)  (Avg)  (Mem, Compute, ID)                                                 
0       4     0x744b,   60148  28.0°C  14.0W  N/A, N/A, 0         0Mhz  96Mhz  20.0%  auto  241.0W  0%     0%    
============================================== End of ROCm SMI Log ===============================================


In [ ]:
# AMD Developer Cloud Validation


In [ ]:
This notebook validates the Solaris Potiguar AI inference pipeline
inside the AMD Developer Cloud.

The production backend is implemented in Ruby on Rails.

This notebook reproduces the same Fireworks AI inference requests
performed by the backend agents.

In [ ]:
import os
import json
import requests
API_KEY = os.getenv("FIREWORKS_API_KEY")

if API_KEY is None:
    raise RuntimeError(
        "Please define FIREWORKS_API_KEY before running this notebook."
    )

In [ ]:
system_prompt = """
You are an expert in solar meteorology.

Analyze ONLY weather information based on real Open-Meteo data.

Consider ALL parameters provided: temperature, cloud cover, solar irradiation,
wind speed, wind direction, humidity, precipitation, rain, showers, snowfall,
UV index, and precipitation probability.

Ignore batteries, consumption and financial aspects.

Return ONLY valid JSON using this exact schema:
{
  "summary":"string",
  "solar_conditions":"HIGH|MEDIUM|LOW",
  "weather_risk":"LOW|MODERATE|HIGH",
  "confidence":0.0-1.0,
  "reasoning":["string"]
}
"""
weather = {
    "temperature": 31,
    "cloud_cover": 12,
    "solar_irradiation": 7.3,
    "wind_speed": 18,
    "wind_direction": 95,
    "humidity": 58,
    "precipitation": 0,
    "rain": 0,
    "showers": 0,
    "snowfall": 0,
    "uv_index": 10,
    "precipitation_probability": 5
}

user_prompt = json.dumps({
    "weather": weather
})

In [ ]:
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

payload = {
    "model": "accounts/fireworks/models/gpt-oss-120b",
    "messages": [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    "temperature": 0.3,
    "max_tokens": 2000
}

response = requests.post(
    "https://api.fireworks.ai/inference/v1/chat/completions",
    headers=headers,
    json=payload,
    timeout=60
)

print("Status:", response.status_code)

result = response.json()

print(result["choices"][0]["message"]["content"])

In [ ]:
## Validation Result

This notebook successfully reproduces the same inference request structure used by the Solaris Potiguar production backend.

The validation confirms:

- execution inside the AMD Developer Cloud;
- availability of AMD ROCm infrastructure;
- successful communication with the Fireworks AI API;
- execution of the GPT-OSS-120B model;
- compatibility between the production multi-agent architecture and the AMD ecosystem.